In [60]:
!pip install langchain_google_genai

In [61]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.messages import HumanMessage

In [62]:
os.environ["GOOGLE_API_KEY"] = "API"

In [63]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [64]:
prompt = """
INSTRUCTION:
Generate a detailed business report of at least 200 words.

CONTEXT:
Company Name: ABC Retail Pvt Ltd
Industry: Retail and E-commerce
Region: India
Business Objective: Expand digital sales and improve customer retention
Current Situation:
- Revenue growth: 12% in last quarter
- Increasing competition from online marketplaces
- Rising logistics and supply chain costs
- Growing customer demand for personalized shopping experiences

ROLE:
You are a Senior Business Strategy Consultant responsible for preparing
professional executive reports for senior leadership teams.

TONE:
Professional, analytical, business-friendly, executive-level,
formal and actionable.

OUTPUT FORMAT:
Generate a report with the following sections:

1. Market Overview
2. Business Opportunities
3. Risks and Challenges
4. Strategic Recommendations
5. Implementation Suggestions
6. Conclusion

CONSTRAINTS / GUARDRAILS:
- Report must be at least 100 words
- Maintain professional business language
- Do not include fictional or unrealistic assumptions
- Provide practical and actionable recommendations
- Keep formatting clean and section-based
- Ensure recommendations align with the provided business context
"""

# Generate report
response = llm.invoke([HumanMessage(content=prompt)])

business_report = response.content

print("Generated Business Report:\n")
print(business_report)

Generated Business Report:

**Business Strategy Report: Enhancing Digital Growth and Customer Retention**

**Company:** ABC Retail Pvt Ltd
**Industry:** Retail and E-commerce
**Region:** India
**Date:** October 26, 2023

---

**1. Market Overview**
The Indian retail and e-commerce landscape is experiencing rapid evolution, characterized by significant digital adoption and intense competition. While ABC Retail Pvt Ltd demonstrated a commendable 12% revenue growth last quarter, this positive trajectory is set against a backdrop of increasing market saturation from established online marketplaces. Consumer expectations are rapidly shifting, with a pronounced demand for highly personalized shopping experiences that extend beyond transactional interactions. This dynamic environment necessitates a strategic pivot to solidify market position and ensure sustainable long-term growth.

**2. Business Opportunities**
Significant opportunities exist for ABC Retail to capitalize on current market tr

In [65]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

# Load model
model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# Split text into sentence-based chunks
def split_text_into_chunks(text, max_chars=400):

    # Split by sentence
    sentences = re.split(r'(?<=[.!?])\s+', text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:

        # Keep adding sentences until limit
        if len(current_chunk) + len(sentence) < max_chars:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


# Translate chunk-by-chunk
chunks = split_text_into_chunks(business_report)

translated_chunks = []

for chunk in chunks:

    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    generated_ids = model.generate(
        **inputs,
        max_length=512,
        num_beams=4,              # better translation quality
        early_stopping=True,
        no_repeat_ngram_size=3    # avoids repeated words
    )

    translated_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    translated_chunks.append(translated_text)

# Combine translated chunks
translated_report = "\n\n".join(translated_chunks)

print("Translated Business Report (Hindi):\n")
print(translated_report)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Translated Business Report (Hindi):

** कौशल रिपोर्ट: डिजिटल वृद्धि और मनपसंद नवीकरण के बारे में।

जबकि ADCC CAL RAL PACANT TAL एक प्रशंसनीय 12% राजस्व वृद्धि दिखाई दी पिछले चौथाई, इस सकारात्मक पैमाने पर बाजार स्थापित बाजार से विकसित बाजारों की एक पीठ के खिलाफ सेट किया जाता है। लोग अपेक्षाएँ तेजी से परिवर्तन कर रहे हैं, और एक बहुत ही निजी खरीदारी के अनुभवों के लिए एक घोषणा की माँग है जो लाभों को बढ़ावा देते हैं।

इस गतिशील वातावरण नेलमानक बाजार स्थिति को ठोस रूप से स्थापित करने के लिए एक रणनीतिक कदम उठाया और सुनिश्चित करने योग्य लंबी वृद्धि. **2. व्यापार महत्वपूर्ण अवसर वर्तमान बाजार के चलन पर राजधानी बनाने के लिए मौजूद है और अपने लक्ष्य प्राप्त करने के अपने डिजिटल बिक्री के अपने लक्ष्य और ग्राहक सुधार.

द न्यू यॉर्क टाइम्स्‌ रिपोर्ट करता है: “एक व्यक्‍तिगत रूप से, एक व्यक्‍ति के लिए यह माँग की जाती है कि वह व्यक्‍तित्व को व्यक्‍त करे और उसे मज़बूत करे ।

अंत-से-दूसरे ग्राहक यात्रा, पोस्ट-चक समर्थन के लिए खोज, उल्लेखनीय रूप से परिवर्तन दरों को बढ़ा सकता है और व्यापार को बढ़ा सकते हैं. 

In [66]:
!pip install langchain langchain-google-genai
!pip install -U langchain langchain-core langchain-google-genai

In [67]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

# Prompt template
summary_prompt = PromptTemplate(
    input_variables=["report"],
    template="""
You are a senior business strategy advisor.

Below is a translated business report.

TASK:
Generate a concise executive summary for senior leadership.

STRICT LANGUAGE RULE:
- Keep the output in the EXACT SAME LANGUAGE as the input report
- Do NOT translate the report into another language
- If input is Hindi, output MUST be in Hindi
- Preserve the source language

Requirements:
- Provide 5 to 7 bullet points
- Highlight major business insights
- Include risks and opportunities
- Include key recommendations
- Keep the language concise and executive-friendly

Business Report:
{report}
"""
)
# Create chain using modern LangChain syntax
chain = summary_prompt | llm

# Generate summary
response = chain.invoke({
    "report": translated_report
})

print("Executive Summary:\n")
print(response.content)

Executive Summary:

**कार्यकारी सारांश: डिजिटल वृद्धि और मनपसंद नवीकरण**

*   ADCC CAL RAL PACANT TAL ने पिछले तिमाही में 12% राजस्व वृद्धि हासिल की है, हालांकि बाजार में व्यक्तिगत खरीदारी अनुभवों की बढ़ती मांग और प्रतिस्पर्धी माहौल मौजूद है।
*   कंपनी के लिए डिजिटल बिक्री और ग्राहक जुड़ाव बढ़ाने का एक महत्वपूर्ण अवसर है, जिसे व्यक्तिगत ग्राहक यात्रा और वफादारी कार्यक्रमों के माध्यम से भुनाया जा सकता है।
*   प्रमुख जोखिमों में बड़े, असंगठित प्रतिस्पर्धियों से बाजार हिस्सेदारी का खतरा और भारत में लॉजिस्टिक्स लागत तथा परिचालन अक्षमताएं शामिल हैं।
*   मुख्य सिफारिशों में डिजिटल कॉमर्स प्लेटफॉर्म को उन्नत करना, मजबूत CRM और व्यक्तिगत वफादारी कार्यक्रमों के माध्यम से निजीकरण बढ़ाना, तथा व्यापक बाजार बुद्धिमत्ता और एनालिटिक्स में निवेश करना शामिल है।
*   अंतिम-मील डिलीवरी और समग्र ग्राहक अनुभव को बेहतर बनाने के लिए आपूर्ति श्रृंखला का समर्थन महत्वपूर्ण है; कार्यान्वयन को चरणबद्ध तरीके से (अगले 3-12 महीने) किया जाना चाहिए, जिसमें क्रॉस-फ़ंक्शनल टीमों और प्रमुख प्रदर्शन संकेतकों की नियमित निगर